<a href="https://colab.research.google.com/github/CoolingVerseOracle/Coolingverse-data/blob/main/05_Risk_Index_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 파일 로드

## 1. 드라이브에서 파일 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# risk_index 테이블 생성

## 성남시

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# 1. 인코딩 및 에러 방지 데이터 로드 함수
def load_csv_safe(file_path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            df = pd.read_csv(file_path, encoding=enc, engine='c')
            print(f"✅ [{file_path}] 읽기 성공")
            return df
        except:
            continue
    return pd.read_csv(file_path, encoding='utf-8-sig', engine='python')

# 2. 분당구 4대 핵심 데이터 로드 (경로 유지)
df_grid = load_csv_safe("/content/drive/MyDrive/grids_bundang_final (1).csv")
df_enf = load_csv_safe("/content/drive/MyDrive/enforcement_with_grid_id_fixed.csv")
df_air = load_csv_safe("/content/drive/MyDrive/air_quality_summary_oracle (1).csv")
df_apt = load_csv_safe("/content/drive/MyDrive/apartments_with_grid_id.csv")

def min_max(series):
    s_min, s_max = series.min(), series.max()
    if s_max == s_min:
        return series * 0.0
    return (series - s_min) / (s_max - s_min + 1e-6)

# =====================================================================
# 🌟 [최적화 1] 단속 또는 아파트가 존재하는 '활성 격자(Active Grid)' 필터링
# =====================================================================
active_grid_ids = pd.concat([df_enf['grid_id'], df_apt['grid_id']]).dropna().unique()
df_active_grids = df_grid[df_grid['grid_id'].isin(active_grid_ids)].copy()
print(f"🎯 전체 {len(df_grid):,}개 격자 중, 분석 대상인 활성 격자 {len(df_active_grids):,}개 추출 완료!")

# =====================================================================
# 🌟 [최종 기획 반영] 수요(단속)와 공급(아파트 유휴면)을 동시 고려한 지수 산출
# =====================================================================
# 1. 활성 격자별 단속 건수 (수요)
enf_grid_counts = df_enf.groupby('grid_id').size().reset_index(name='enf_count')

# 2. 활성 격자별 아파트 유휴 주차면 합계 (공급)
apt_open_counts = df_apt.groupby('grid_id')['open_count'].sum().reset_index(name='total_open_count')

# 3. Base 테이블 병합
df_base = df_active_grids[['grid_id', 'center_lat', 'center_lng']].merge(
    enf_grid_counts, on='grid_id', how='left'
).merge(
    apt_open_counts, on='grid_id', how='left'
).fillna({'enf_count': 0, 'total_open_count': 0})

# 4. 수요 압박 및 공급 부족 지수 산출 (Log 변환 및 역수 적용)
df_base['demand_pressure'] = min_max(np.log1p(df_base['enf_count']))
df_base['supply_shortage'] = 1.0 - min_max(df_base['total_open_count']) # 빈자리가 적을수록 1.0에 근접

# =====================================================================
# 🌟 [오류 수정 및 최적화 2] 24시 표기 오류 완벽 해결 및 측정소 매핑 (cKDTree)
# =====================================================================
print("🔄 대기질 시간 파싱 및 최단 거리 측정소 매핑 중...")

# [에러 픽스] 24:00:00 을 00:00:00 으로 치환하여 파싱 에러 원천 차단
df_air['measured_at_fixed'] = df_air['measured_at'].astype(str).str.replace(' 24:00:00', ' 00:00:00').str.replace(' 24:00', ' 00:00')
df_air['hour'] = pd.to_datetime(df_air['measured_at_fixed']).dt.hour

air_grid_h = df_air.groupby(['grid_id', 'hour'])[['no2', 'co']].mean().reset_index()
air_overall_h = df_air.groupby('hour')[['no2', 'co']].mean().reset_index()

station_info = df_air[['grid_id', 'lat', 'lng']].drop_duplicates()
# 좌표 결측치 방어
station_info = station_info.dropna(subset=['lat', 'lng'])
tree = cKDTree(station_info[['lat', 'lng']].values)

distances, indices = tree.query(df_base[['center_lat', 'center_lng']].values)
df_base['nearest_station_grid_id'] = station_info['grid_id'].values[indices]

# =====================================================================
# 🌟 [최적화 3] Vectorization 교차 조인 및 최종 위험 지수(Risk Score) 산출
# =====================================================================
print("🔄 위험지수(Risk Index) 24시간 매트릭스 일괄 연산 중...")
traffic_weights = pd.DataFrame(list({
    0:0.7, 1:0.7, 2:0.7, 3:0.7, 4:0.7, 5:0.7, 6:0.8,
    7:1.3, 8:1.3, 9:1.3, 10:1.0, 11:1.0, 12:1.0, 13:1.0, 14:1.0, 15:1.0, 16:1.0,
    17:1.3, 18:1.3, 19:1.3, 20:1.3, 21:0.8, 22:0.7, 23:0.7
}.items()), columns=['hour', 'tw'])

# 1. 24시간 확장 (Cross Join)
df_expanded = df_base.merge(pd.DataFrame({'hour': range(24)}), how='cross')
df_expanded = df_expanded.merge(traffic_weights, on='hour', how='left')

# 2. 교통 혼잡도 (TC) 계산
df_expanded['tc'] = np.minimum(1.0, df_expanded['demand_pressure'] * (df_expanded['tw'] / 1.3))

# 3. 내 격자에 매핑된 측정소의 대기질 병합 및 결측치 이중 방어
df_expanded = df_expanded.merge(
    air_grid_h,
    left_on=['nearest_station_grid_id', 'hour'],
    right_on=['grid_id', 'hour'],
    how='left',
    suffixes=('', '_air')
)
df_expanded = df_expanded.merge(air_overall_h, on='hour', how='left', suffixes=('', '_overall'))
df_expanded['no2'] = df_expanded['no2'].fillna(df_expanded['no2_overall'])
df_expanded['co'] = df_expanded['co'].fillna(df_expanded['co_overall'])

# 4. 환경 민감도 (ES) 및 최종 Risk Score 계산
df_expanded['es'] = np.minimum(1.0, (df_expanded['no2'] / 0.075) * 0.6 + (df_expanded['co'] / 1.79) * 0.4)
df_expanded['risk_score'] = (0.35 * df_expanded['supply_shortage'] +
                             0.25 * df_expanded['demand_pressure'] +
                             0.15 * df_expanded['tc'] +
                             0.25 * df_expanded['es']) * 100.0

# =====================================================================
# 5. 최종 데이터 프레임 포맷팅 및 추출
# =====================================================================
batch_date_str = pd.Timestamp.now(tz='Asia/Seoul').strftime('%Y-%m-%d')

df_risk_final = pd.DataFrame({
    'grid_id': df_expanded['grid_id'],
    'hour_of_day': df_expanded['hour'],
    'demand_pressure': df_expanded['demand_pressure'].round(4),
    'supply_shortage': df_expanded['supply_shortage'].round(4),
    'traffic_congest': df_expanded['tc'].round(4),
    'env_sensitivity': df_expanded['es'].round(4),
    'risk_score': df_expanded['risk_score'].round(2),
    'batch_date': batch_date_str
})

df_risk_final.insert(0, 'risk_id', range(1, len(df_risk_final) + 1))
output_file = "/content/drive/MyDrive/bundang_risk_index_final_v2.csv"
df_risk_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"🎉 에러 완벽 해결! 분당구 B2G 시뮬레이션 지수 산출 완료! ({output_file})")

✅ [/content/drive/MyDrive/grids_bundang_final (1).csv] 읽기 성공
✅ [/content/drive/MyDrive/enforcement_with_grid_id_fixed.csv] 읽기 성공
✅ [/content/drive/MyDrive/air_quality_summary_oracle (1).csv] 읽기 성공
✅ [/content/drive/MyDrive/apartments_with_grid_id.csv] 읽기 성공
🎯 전체 122,318개 격자 중, 분석 대상인 활성 격자 1,306개 추출 완료!
🔄 대기질 시간 파싱 및 최단 거리 측정소 매핑 중...
🔄 위험지수(Risk Index) 24시간 매트릭스 일괄 연산 중...
🎉 에러 완벽 해결! 분당구 B2G 시뮬레이션 지수 산출 완료! (/content/drive/MyDrive/bundang_risk_index_final_v2.csv)


## 부천시

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# 1. 인코딩 및 에러 방지 데이터 로드 함수
def load_csv_safe(file_path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            df = pd.read_csv(file_path, encoding=enc, engine='c')
            print(f"✅ [{file_path}] 읽기 성공")
            return df
        except:
            continue
    return pd.read_csv(file_path, encoding='utf-8-sig', engine='python')

# 2. 방금 전까지 완벽하게 정제한 부천시 4대 핵심 데이터 로드
df_grid = load_csv_safe("/content/drive/MyDrive/grids_bucheon_final.csv") # 부천시 최종 그리드
df_enf = load_csv_safe("/content/drive/MyDrive/부천시_단속위치_grid_mapped.csv") # 부천시 단속 데이터
df_apt = load_csv_safe("/content/drive/MyDrive/bucheon_apartments_erd.csv") # 부천시 아파트 데이터
df_air = load_csv_safe("/content/drive/MyDrive/bucheon_air_quality_grid_mapped.csv") # 부천시 대기질 데이터

def min_max(series):
    s_min, s_max = series.min(), series.max()
    if s_max == s_min:
        return series * 0.0
    return (series - s_min) / (s_max - s_min + 1e-6)

# =====================================================================
# 🌟 [최적화 1] 단속 또는 아파트가 존재하는 '활성 격자(Active Grid)' 필터링
# =====================================================================
active_grid_ids = pd.concat([df_enf['grid_id'], df_apt['grid_id']]).dropna().unique()
df_active_grids = df_grid[df_grid['grid_id'].isin(active_grid_ids)].copy()
print(f"🎯 전체 {len(df_grid):,}개 격자 중, 분석 대상인 활성 격자 {len(df_active_grids):,}개 추출 완료!")

# =====================================================================
# 🌟 [최종 기획 반영] 수요(단속)와 공급(아파트 유휴면)을 동시 고려한 지수 산출
# =====================================================================
# 1. 활성 격자별 단속 건수 (수요)
enf_grid_counts = df_enf.groupby('grid_id').size().reset_index(name='enf_count')

# 2. 활성 격자별 아파트 유휴 주차면 합계 (공급)
apt_open_counts = df_apt.groupby('grid_id')['open_count'].sum().reset_index(name='total_open_count')

# 3. Base 테이블 병합
df_base = df_active_grids[['grid_id', 'center_lat', 'center_lng']].merge(
    enf_grid_counts, on='grid_id', how='left'
).merge(
    apt_open_counts, on='grid_id', how='left'
).fillna({'enf_count': 0, 'total_open_count': 0})

# 4. 수요 압박 및 공급 부족 지수 산출 (Log 변환 및 역수 적용)
df_base['demand_pressure'] = min_max(np.log1p(df_base['enf_count']))
df_base['supply_shortage'] = 1.0 - min_max(df_base['total_open_count']) # 빈자리가 적을수록 1.0에 근접

# =====================================================================
# 🌟 [최적화 2] 1시간 단위 대기질 시간 파싱 및 가장 가까운 측정소 매핑 (cKDTree)
# =====================================================================
# measured_at 포맷: "2025-01-02 01:00" -> 시간만 추출
df_air['hour'] = pd.to_datetime(df_air['measured_at']).dt.hour
air_grid_h = df_air.groupby(['grid_id', 'hour'])[['no2', 'co']].mean().reset_index()
air_overall_h = df_air.groupby('hour')[['no2', 'co']].mean().reset_index()

print("🔄 활성 격자에 가장 가까운 대기질 측정소를 매핑합니다...")
station_info = df_air[['grid_id', 'lat', 'lng']].drop_duplicates()
tree = cKDTree(station_info[['lat', 'lng']].values)

distances, indices = tree.query(df_base[['center_lat', 'center_lng']].values)
df_base['nearest_station_grid_id'] = station_info['grid_id'].values[indices]

# =====================================================================
# 🌟 [최적화 3] Vectorization 교차 조인 및 최종 위험 지수(Risk Score) 산출
# =====================================================================
print("🔄 위험지수(Risk Index) 24시간 매트릭스 연산 중...")
traffic_weights = pd.DataFrame(list({
    0:0.7, 1:0.7, 2:0.7, 3:0.7, 4:0.7, 5:0.7, 6:0.8,
    7:1.3, 8:1.3, 9:1.3, 10:1.0, 11:1.0, 12:1.0, 13:1.0, 14:1.0, 15:1.0, 16:1.0,
    17:1.3, 18:1.3, 19:1.3, 20:1.3, 21:0.8, 22:0.7, 23:0.7
}.items()), columns=['hour', 'tw'])

# 1. 24시간 확장 (Cross Join)
df_expanded = df_base.merge(pd.DataFrame({'hour': range(24)}), how='cross')
df_expanded = df_expanded.merge(traffic_weights, on='hour', how='left')

# 2. 교통 혼잡도 (TC) 계산
df_expanded['tc'] = np.minimum(1.0, df_expanded['demand_pressure'] * (df_expanded['tw'] / 1.3))

# 3. 내 격자에 매핑된 측정소의 대기질 병합 및 결측치 이중 방어
df_expanded = df_expanded.merge(
    air_grid_h,
    left_on=['nearest_station_grid_id', 'hour'],
    right_on=['grid_id', 'hour'],
    how='left',
    suffixes=('', '_air')
)
df_expanded = df_expanded.merge(air_overall_h, on='hour', how='left', suffixes=('', '_overall'))
df_expanded['no2'] = df_expanded['no2'].fillna(df_expanded['no2_overall'])
df_expanded['co'] = df_expanded['co'].fillna(df_expanded['co_overall'])

# 4. 환경 민감도 (ES) 및 최종 Risk Score 계산
df_expanded['es'] = np.minimum(1.0, (df_expanded['no2'] / 0.075) * 0.6 + (df_expanded['co'] / 1.79) * 0.4)
df_expanded['risk_score'] = (0.35 * df_expanded['supply_shortage'] +
                             0.25 * df_expanded['demand_pressure'] +
                             0.15 * df_expanded['tc'] +
                             0.25 * df_expanded['es']) * 100.0

# =====================================================================
# 5. 최종 데이터 프레임 포맷팅 및 추출
# =====================================================================
batch_date_str = pd.Timestamp.now(tz='Asia/Seoul').strftime('%Y-%m-%d')

df_risk_final = pd.DataFrame({
    'grid_id': df_expanded['grid_id'],
    'hour_of_day': df_expanded['hour'],
    'demand_pressure': df_expanded['demand_pressure'].round(4),
    'supply_shortage': df_expanded['supply_shortage'].round(4),
    'traffic_congest': df_expanded['tc'].round(4),
    'env_sensitivity': df_expanded['es'].round(4),
    'risk_score': df_expanded['risk_score'].round(2),
    'batch_date': batch_date_str
})

df_risk_final.insert(0, 'risk_id', range(1, len(df_risk_final) + 1))
output_file = "/content/drive/MyDrive/bucheon_risk_index_final.csv"
df_risk_final.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"🎉 부천시 타겟 B2G 시뮬레이션 지수 산출 완료! ({output_file})")

✅ [/content/drive/MyDrive/grids_bucheon_final.csv] 읽기 성공
✅ [/content/drive/MyDrive/부천시_단속위치_grid_mapped.csv] 읽기 성공
✅ [/content/drive/MyDrive/bucheon_apartments_erd.csv] 읽기 성공
✅ [/content/drive/MyDrive/bucheon_air_quality_grid_mapped.csv] 읽기 성공
🎯 전체 55,256개 격자 중, 분석 대상인 활성 격자 2,264개 추출 완료!
🔄 활성 격자에 가장 가까운 대기질 측정소를 매핑합니다...
🔄 위험지수(Risk Index) 24시간 매트릭스 연산 중...
🎉 부천시 타겟 B2G 시뮬레이션 지수 산출 완료! (/content/drive/MyDrive/bucheon_risk_index_final.csv)
